In [2]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_kf_oof = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_kf_oof_train.parquet')

print(train_kf_oof.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 40)


In [10]:
# ============================================================
# 1. 전체 OOF 종속변수 구간별 RMSE 분석
# ============================================================

import numpy as np

# 종속변수 분위수 기준으로 구간 설정
quantiles = [0, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0]
labels = ['0~50%', '50~75%', '75~90%', '90~95%', '95~99%', '99~100%']

train_kf_oof['y_kf_val_quant'] = pd.qcut(  # pd.qcut() 지정한 분위수 경계값 기준으로 구간 나눠줌
    train_kf_oof['y_kf_val'],  # 매개변수: 구간 나눌 데이터
    q=quantiles,  # 매개변수: 구간을 나눌 위치(분위수)
    labels=labels # 매개변수: 각각의 라벨(q보다 하나 적어야 함)
)

# 구간별 RMSE 계산
y_kf_val_segment_rmse = (
    train_kf_oof
    .groupby('y_kf_val_quant', observed=True)  # observed=True: 관측된 행만 기재(count가 0인 행은 제거)
    .agg(
        count=('resid_sq', 'size'),
        mse=('resid_sq', 'mean'),  # 잔차제곱의 평균 = mse, 루트하면 rmse
        resid_sq_sum=('resid_sq', 'sum')
    )
)

y_kf_val_segment_rmse['rmse'] = np.sqrt(y_kf_val_segment_rmse['mse'])
y_kf_val_segment_rmse['resid_sq_sum_contribution_pct(%)'] = (
    y_kf_val_segment_rmse['resid_sq_sum']/
    y_kf_val_segment_rmse['resid_sq_sum'].sum()
    *100
)

display(y_kf_val_segment_rmse[['count', 'rmse', 'resid_sq_sum','resid_sq_sum_contribution_pct(%)']])

,count,rmse,resid_sq_sum,resid_sq_sum_contribution_pct(%)
y_kf_val_quant,,,,
0~50%,340940,0.096568,3179.427499,9.516899
50~75%,127705,0.094845,1148.784463,3.438627
75~90%,87488,0.157642,2174.173762,6.507899
90~95%,29345,0.280564,2309.926209,6.914243
95~99%,24674,0.655039,10587.012558,31.689834
99~100%,6141,1.510367,14008.905566,41.932498


In [17]:
# ============================================================
# 2. Fold별 종속변수 구간별 RMSE 분석
# ============================================================

# Fold별, 종속변수 구간별 RMSE 계산
fold_segment_rmse = (
    train_kf_oof
    .groupby(['fold_num', 'y_kf_val_quant'], observed=True)
    .agg(
        count=('resid_sq', 'size'),
        mse=('resid_sq', 'mean')
    )
)

fold_segment_rmse['rmse'] = np.sqrt(fold_segment_rmse['mse'])

# Fold별 RMSE 비교표
fold_segment_rmse_pivot = fold_segment_rmse['rmse'].unstack('fold_num')

display(fold_segment_rmse_pivot)

fold_num,1,2,3,4,5
y_kf_val_quant,,,,,
0~50%,0.126872,0.084618,0.087797,0.086416,0.089836
50~75%,0.118810,0.091043,0.087636,0.090398,0.089004
75~90%,0.189627,0.148832,0.141992,0.160345,0.141922
90~95%,0.316162,0.275802,0.263280,0.270517,0.261576
95~99%,0.697080,0.546779,0.641688,0.678735,0.625992
99~100%,1.453365,0.816463,1.568512,1.899016,1.374769


In [19]:
train_kf_oof.head(20)

,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,imerg_rain_24h,sfc_rain_1h,sfc_rain_6h,sfc_rain_24h,radar_reflectivity_mean,radar_reflectivity_p95,radar_reflectivity_ge20_fraction,cat_area,wsarea,mean_elevation,mean_slope,urban_ratio,forest_ratio,stream_order,target_maxrise_6h,event_id,wl_diff_1h,wl_diff_3h,wl_diff_6h,wl_diff_12h,wl_diff_24h,imerg_int_1to6h,imerg_int_6to24h,sfc_int_1to6h,sfc_int_6to24h,y_kf_val,y_kf_pred,resid,resid_sq,fold_num,y_kf_val_quant
0,2100-01-01,ST0001,0.29,0.29,0.28,0.28,0.28,0.27,0.000,0.000,0.000,0.0,0.0,0.0,12.162,25.274,0.1445,2.8605,37.0279,124.37,9.6147,11.1141,52.4597,1.9592,0.00,1,0.00,0.01,0.01,0.01,0.02,0.0000,0.000000,0.00,0.000000,0.00,0.024048,-0.024048,0.000578,5,0~50%
1,2100-01-01,ST0002,0.77,0.76,0.77,0.77,0.76,0.77,0.000,0.000,0.000,0.0,0.0,0.0,0.912,12.060,0.0000,3.9564,42.8163,180.09,16.5687,2.5710,88.0622,1.8000,0.00,1,0.01,0.00,0.00,0.01,0.00,0.0000,0.000000,0.00,0.000000,0.00,0.003616,-0.003616,0.000013,5,0~50%
2,2100-01-01,ST0003,0.73,0.73,0.74,0.74,0.73,0.73,0.000,0.000,0.000,0.0,0.0,0.0,18.060,30.920,0.5654,4.2240,80.8559,125.04,12.1864,34.3816,56.6211,2.2500,0.00,1,0.00,-0.01,-0.01,0.00,0.00,0.0000,0.000000,0.00,0.000000,0.00,0.014925,-0.014925,0.000223,5,0~50%
3,2100-01-01,ST0004,1.01,1.01,1.01,1.01,1.01,1.01,0.000,0.000,0.000,0.0,0.0,0.0,0.029,0.000,0.0000,3.2098,252.5946,199.30,13.5951,2.7206,73.1707,2.1207,0.00,1,0.00,0.00,0.00,0.00,0.00,0.0000,0.000000,0.00,0.000000,0.00,-0.005762,0.005762,0.000033,5,0~50%
4,2100-01-01,ST0005,1.01,1.01,1.01,1.01,1.03,1.06,3.018,7.493,7.497,0.0,0.0,0.0,6.898,21.500,0.0811,3.6470,118.5237,246.72,17.1248,0.4779,88.0002,2.0000,0.03,1,0.00,0.00,0.00,-0.02,-0.05,0.8950,0.000222,0.00,0.000000,0.03,0.039088,-0.009088,0.000083,5,50~75%
5,2100-01-01,ST0006,0.75,0.75,0.74,0.77,0.73,0.88,0.000,0.144,0.146,0.0,2.0,2.0,15.808,21.178,0.0811,11.8928,14.9987,38.11,5.7671,63.7535,18.7559,1.2222,0.08,1,0.00,0.01,-0.02,0.02,-0.13,0.0288,0.000111,0.40,0.000000,0.08,0.105763,-0.025763,0.000664,5,75~90%
6,2100-01-01,ST0007,1.58,1.58,1.58,1.58,1.60,1.64,2.621,4.574,6.215,0.0,2.0,2.0,19.108,27.750,0.3613,2.9860,99.0096,335.78,16.2836,1.0168,87.6109,2.0588,0.00,1,0.00,0.00,0.00,-0.02,-0.06,0.3906,0.091167,0.40,0.000000,0.00,0.079097,-0.079097,0.006256,5,0~50%
7,2100-01-01,ST0008,122.99,123.00,122.98,122.97,123.00,123.00,1.496,15.696,16.742,0.0,0.0,0.0,21.069,24.540,0.7139,6.0509,993.3904,274.74,16.9880,0.9004,81.7343,2.4133,0.03,1,-0.01,0.01,0.02,-0.01,-0.01,2.8400,0.058111,0.00,0.000000,0.03,-0.024613,0.054613,0.002983,5,50~75%
8,2100-01-01,ST0009,1.41,1.41,1.42,1.42,1.43,1.45,0.125,0.184,0.184,0.0,0.0,0.0,0.039,0.000,0.0000,3.7874,87.1406,45.85,5.5734,12.3710,18.9725,2.0370,0.00,1,0.00,-0.01,-0.01,-0.02,-0.04,0.0118,0.000000,0.00,0.000000,0.00,0.018924,-0.018924,0.000358,5,0~50%
9,2100-01-01,ST0010,1.68,1.69,1.71,1.74,1.80,1.32,0.000,0.013,0.121,0.0,0.0,0.0,0.000,0.000,0.0000,3.4582,934.2290,501.16,23.1930,2.9555,88.8847,2.2703,-0.01,1,-0.01,-0.03,-0.06,-0.12,0.36,0.0026,0.006000,0.00,0.000000,-0.01,0.017734,-0.027734,0.000769,5,0~50%


In [22]:
# ============================================================
# 4. 종속변수 상위 5% 구간의 고잔차 데이터 분석
# ============================================================

# 종속변수 상위 5% 데이터 추출
high_y_resid = train_kf_oof.loc[
    train_kf_oof['y_kf_val_quant'].isin(['95~99%', '99~100%'])
].copy()

# 제곱잔차 기준 내림차순 정렬
high_y_error = high_y_resid.sort_values(
    'resid_sq', ascending=False
)

# # 주요 컬럼 확인
# display(
#     high_y_error[
#         ['fold', 'station_id', 'y_kf_val', 'y_kf_pred',
#          'residual', 'residual_sq']
#     ]
# )

display(high_y_error['station_id'].value_counts())

station_id
ST0358    830
ST0326    812
ST0292    810
ST0159    780
ST0347    750
ST0110    679
ST0313    637
ST0415    437
ST0234    334
ST0130    286
ST0275    274
ST0129    272
ST0067    265
ST0192    241
ST0051    228
ST0380    206
ST0335    193
ST0164    190
ST0178    169
ST0282    168
ST0028    166
ST0310    165
ST0388    163
ST0221    158
ST0062    149
ST0149    148
ST0354    145
ST0332    142
ST0297    141
ST0148    133
ST0353    132
ST0393    132
ST0194    130
ST0095    129
ST0428    128
ST0357    127
ST0397    125
ST0356    125
ST0116    125
ST0424    123
ST0272    123
ST0059    122
ST0318    115
ST0242    114
ST0336    113
ST0226    111
ST0323    111
ST0013    110
ST0040    109
ST0036    109
ST0073    109
ST0435    105
ST0006    103
ST0301    103
ST0268    102
ST0258    102
ST0303    101
ST0125    100
ST0207    100
ST0093     96
ST0312     96
ST0337     95
ST0084     94
ST0368     93
ST0260     93
ST0092     93
ST0413     92
ST0101     92
ST0152     91
ST0007     90
ST0345   